# Colab Indexing For FPT RAG

Notebook nay dung `google/embeddinggemma-300m` tren Google Colab GPU T4 de embed va index truc tiep len Qdrant Cloud.

Muc tieu:
- Tai dataset truc tiep tu Hugging Face: `huyvux3005/flm_dataset`
- Tao vector 768-dim tu `google/embeddinggemma-300m`
- Upsert vao 2 collection Qdrant
- Ghi BM25 artifacts dung format ma `my_project` dang doc
- Sau khi index xong, quay lai local de chay `query` va `eval`


## 1. Install dependencies
Chay cell nay tren Colab T4 dau tien.


In [ ]:
!pip -q install qdrant-client rank-bm25 transformers sentencepiece accelerate torch datasets huggingface_hub


## 2. Config
Nhap thong tin dataset Hugging Face va Qdrant Cloud.


In [ ]:
from pathlib import Path

HF_DATASET_REPO = 'huyvux3005/flm_dataset'
ARTIFACTS_DIR = Path('/content/artifacts')
QDRANT_URL = 'https://feb5e97e-774a-4277-a7d4-b6421fea6ac8.us-east4-0.gcp.cloud.qdrant.io'
QDRANT_API_KEY = 'REPLACE_ME'
CURRICULUM_COLLECTION = 'fpt_curriculum_map_v1'
SYLLABUS_COLLECTION = 'fpt_syllabus_detail_v1'
EMBEDDING_MODEL_NAME = 'google/embeddinggemma-300m'
EMBEDDING_MODEL_VERSION = 'google-embeddinggemma-300m-colab-v1'
BATCH_SIZE = 256
UPLOAD_BATCH_SIZE = 256
VECTOR_SIZE = 768
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Helpers
Doc dataset tu Hugging Face, BM25 tokenization, point id, va Qdrant helpers giong code local.


In [ ]:
import hashlib
import json
import re
from datetime import date, datetime
from typing import Any

import torch
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.http import models
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

TOKEN_RE = re.compile(r'[A-Za-z0-9_.-]+')


def dataset_split_to_rows(split) -> list[dict[str, Any]]:
    return [dict(row) for row in split]


def load_dataset_bundle(dataset_repo: str) -> dict[str, list[dict[str, Any]]]:
    curriculum = dataset_split_to_rows(load_dataset(dataset_repo, data_files={'train': 'curriculum_semester_chunks.jsonl'}, split='train'))
    offering_map = dataset_split_to_rows(load_dataset(dataset_repo, data_files={'train': 'offering_map.jsonl'}, split='train'))
    canonical_syllabus = dataset_split_to_rows(load_dataset(dataset_repo, data_files={'train': 'canonical_syllabus.jsonl'}, split='train'))
    syllabus_detail = dataset_split_to_rows(load_dataset(dataset_repo, data_files={'train': 'syllabus_detail_chunks.jsonl'}, split='train'))
    return {
        'curriculum': curriculum,
        'offering_map': offering_map,
        'canonical_syllabus': canonical_syllabus,
        'syllabus_detail': syllabus_detail,
    }


def tokenize(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_RE.findall(text)]


def build_index_records(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    records = []
    for row in rows:
        payload = dict(row)
        payload['embedding_model_version'] = EMBEDDING_MODEL_VERSION
        records.append({'id': row['chunk_id'], 'text': row['text'], 'payload': payload})
    return records


def point_id(chunk_id: str) -> int:
    return int(hashlib.sha1(chunk_id.encode('utf-8')).hexdigest()[:15], 16)


def ensure_collection(client: QdrantClient, collection_name: str) -> None:
    existing = {c.name for c in client.get_collections().collections}
    if collection_name not in existing:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE),
        )
    for field_name, schema in {
        'chunk_id': models.PayloadSchemaType.KEYWORD,
        'course_id': models.PayloadSchemaType.KEYWORD,
        'major_code': models.PayloadSchemaType.KEYWORD,
        'semester': models.PayloadSchemaType.INTEGER,
        'combo_short_name': models.PayloadSchemaType.KEYWORD,
        'chunk_type': models.PayloadSchemaType.KEYWORD,
        'syllabus_id': models.PayloadSchemaType.KEYWORD,
        'canonical_syllabus_key': models.PayloadSchemaType.KEYWORD,
        'parent_offering_key': models.PayloadSchemaType.KEYWORD,
    }.items():
        try:
            client.create_payload_index(collection_name, field_name, schema)
        except Exception:
            pass


def _json_safe(value: Any) -> Any:
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def save_bm25(records: list[dict[str, Any]], output_path: Path) -> None:
    tokenized = [tokenize(doc['text']) for doc in records]
    _ = BM25Okapi(tokenized or [['empty']])
    with output_path.open('w', encoding='utf-8') as handle:
        json.dump(
            {'docs': records, 'tokenized_corpus': tokenized},
            handle,
            ensure_ascii=False,
            default=_json_safe,
        )


## 4. Load EmbeddingGemma
Load model tren GPU T4 va dung mean pooling + L2 normalize de ra vector 768 chieu.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME, torch_dtype=torch.float32)
model.to(device)
model.eval()
print({'device': device, 'model': EMBEDDING_MODEL_NAME})

@torch.inference_mode()
def embed_texts(texts: list[str], batch_size: int = BATCH_SIZE, max_length: int = 512) -> list[list[float]]:
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        outputs = model(**encoded)
        last_hidden = outputs.last_hidden_state.float()
        attention_mask = encoded['attention_mask']

        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        pooled = summed / counts
        pooled = torch.nan_to_num(pooled, nan=0.0, posinf=0.0, neginf=0.0)

        norms = torch.linalg.norm(pooled, dim=1, keepdim=True).clamp(min=1e-12)
        pooled = pooled / norms
        pooled = torch.nan_to_num(pooled, nan=0.0, posinf=0.0, neginf=0.0)

        all_embeddings.extend(pooled.cpu().tolist())
        print(f'embedded {min(start + len(batch), len(texts))}/{len(texts)}')

    return all_embeddings


## 5. Build records and connect Qdrant


In [ ]:
bundle = load_dataset_bundle(HF_DATASET_REPO)
curriculum_records = build_index_records(bundle['curriculum'])
syllabus_records = build_index_records(bundle['syllabus_detail'])
print({'dataset_repo': HF_DATASET_REPO, 'curriculum_records': len(curriculum_records), 'syllabus_records': len(syllabus_records)})

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
ensure_collection(client, CURRICULUM_COLLECTION)
ensure_collection(client, SYLLABUS_COLLECTION)
print('collections ensured')


## 6. Embed and upload
Cell nay la cell chinh de index len Qdrant Cloud.


In [ ]:
import math

def upload_records(client: QdrantClient, collection_name: str, records: list[dict[str, Any]], batch_size: int = UPLOAD_BATCH_SIZE):
    texts = [record['text'] for record in records]
    vectors = embed_texts(texts, batch_size=batch_size)

    if len(vectors) != len(records):
        raise ValueError(f'Vector count mismatch: {len(vectors)} != {len(records)}')

    for start in range(0, len(records), batch_size):
        rec_batch = records[start:start + batch_size]
        vec_batch = vectors[start:start + batch_size]
        points = []

        for record, vector in zip(rec_batch, vec_batch):
            if len(vector) != VECTOR_SIZE:
                raise ValueError(f"Invalid vector size for {record['id']}: {len(vector)} != {VECTOR_SIZE}")

            clean_vector = [float(x) for x in vector]
            if not all(math.isfinite(x) for x in clean_vector):
                raise ValueError(f"Non-finite values detected in vector for {record['id']}")

            payload = dict(record['payload'])
            payload.setdefault('chunk_id', record['id'])

            points.append(
                models.PointStruct(
                    id=point_id(record['id']),
                    vector=clean_vector,
                    payload=payload,
                )
            )

        client.upsert(
            collection_name=collection_name,
            points=points,
            wait=True,
        )
        print(f"uploaded {collection_name}: {min(start + len(rec_batch), len(records))}/{len(records)}")

upload_records(client, CURRICULUM_COLLECTION, curriculum_records, batch_size=UPLOAD_BATCH_SIZE)
upload_records(client, SYLLABUS_COLLECTION, syllabus_records, batch_size=UPLOAD_BATCH_SIZE)
print('qdrant indexing done')


## 7. Save BM25 artifacts
Download 2 file JSON nay ve local va dat vao `my_project/artifacts/`.


In [ ]:
bm25_curriculum_path = ARTIFACTS_DIR / 'bm25_curriculum_map.json'
bm25_syllabus_path = ARTIFACTS_DIR / 'bm25_syllabus_detail.json'
save_bm25(curriculum_records, bm25_curriculum_path)
save_bm25(syllabus_records, bm25_syllabus_path)
print({'bm25_curriculum': str(bm25_curriculum_path), 'bm25_syllabus': str(bm25_syllabus_path)})


## 8. Quick verification
Kiem tra so luong point da len cloud.


In [ ]:
print(client.count(CURRICULUM_COLLECTION, exact=True).count)
print(client.count(SYLLABUS_COLLECTION, exact=True).count)


## 9. Sau khi notebook chay xong
Tren may local:
1. Copy hai BM25 artifact ve `my_project/artifacts/`
2. Set `QDRANT_URL`, `QDRANT_API_KEY` trung voi cloud
3. Chay lai:
   - `uv run python main.py query --text "AI ky 4 hoc gi?"`
   - `uv run python main.py query --text "Schedule cua LAB211 la gi?"`
   - `uv run python main.py eval`
